In [0]:
%sql
use catalog deltalake_catalog;

In [0]:
%sql
DROP TABLE IF EXISTS demo_taxi_200files;
CREATE TABLE demo_taxi_200files (
  vendor_id STRING,
  pickup_datetime TIMESTAMP,
  dropoff_datetime TIMESTAMP,
  passenger_count INT,
  trip_distance DOUBLE,
  pickup_longitude DOUBLE,
  pickup_latitude DOUBLE,
  rate_code_id INT,
  store_and_fwd_flag STRING,
  dropoff_longitude DOUBLE,
  dropoff_latitude DOUBLE,
  payment_type STRING,
  fare_amount DOUBLE,
  extra DOUBLE,
  mta_tax DOUBLE,
  tip_amount DOUBLE,
  tolls_amount DOUBLE,
  total_amount DOUBLE
)
USING DELTA
TBLPROPERTIES (
  delta.autoOptimize.optimizeWrite = false,
  delta.autoOptimize.autoCompact = false
);

In [0]:
# Find the table location from DESCRIBE DETAIL first:
table_location = "dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow"

# List data files (ignore _delta_log)
all_files = [f.path for f in dbutils.fs.ls(table_location) if f.path.endswith(".parquet")]

# Pick first 10 files
files10 = all_files[:10]
print("Using these 10 files:", files10)

# Read them as Parquet
df_10 = spark.read.parquet(*files10)

# Repartition into 200 files and write into your demo Delta table
df_10.repartition(200).write.format("delta").mode("overwrite").saveAsTable("demo_taxi_200files")

Using these 10 files: ['dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-7dcc09ea-2fc1-491d-a234-3b0ce1db9336-c002.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-812df468-c3fb-405d-84d6-32cb249d8db9-c003.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-a4b4b3de-4231-4c0e-88d6-c14f202ff232-c000.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-b3d2db79-4a87-4d35-8a20-a02725fb655f-c001.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-158e21e1-9d7b-44e9-ad15-3ba7e1797de8-c002.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-9072b615-4da5-4c0c-b2c0-83f08d1944ac-c000.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-9baf2d71-1e4d-49a0-a131-03c825853637-c003.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-d61d36d1-b864-4dc2-b

In [0]:
%sql
select count(*) from demo_taxi_200files where trip_distance > 100;

count(*)
209


In [0]:
%sql
ALTER TABLE demo_taxi_200files CLUSTER BY (trip_distance);

In [0]:
%sql
describe history demo_taxi_200files;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2025-09-26T13:53:13.000Z,147711536460494,sumit.trendytech03@outlook.com,CLUSTER BY,"Map(oldClusteringColumns -> , newClusteringColumns -> trip_distance)",null,List(1701835403252703),0926-134922-hibp07z1-v2n,3,WriteSerializable,true,Map(),null,Databricks-Runtime/17.1.x-photon-scala2.13
3,2025-09-26T13:53:12.000Z,147711536460494,sumit.trendytech03@outlook.com,ROW TRACKING BACKFILL,Map(batchId -> 0),null,List(1701835403252703),0926-134922-hibp07z1-v2n,2,SnapshotIsolation,false,Map(),null,Databricks-Runtime/17.1.x-photon-scala2.13
2,2025-09-26T13:53:09.000Z,147711536460494,sumit.trendytech03@outlook.com,UPGRADE PROTOCOL,"Map(newProtocol -> {""minReaderVersion"":3,""minWriterVersion"":7,""readerFeatures"":[""deletionVectors""],""writerFeatures"":[""deletionVectors"",""domainMetadata"",""rowTracking"",""invariants"",""appendOnly""]})",null,List(1701835403252703),0926-134922-hibp07z1-v2n,1,WriteSerializable,true,Map(),null,Databricks-Runtime/17.1.x-photon-scala2.13
1,2025-09-26T13:51:06.000Z,147711536460494,sumit.trendytech03@outlook.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.autoOptimize.autoCompact"":""false"",""delta.autoOptimize.optimizeWrite"":""false"",""collation"":""UTF8_BINARY"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1701835403252703),0926-134922-hibp07z1-v2n,0,WriteSerializable,false,"Map(numFiles -> 200, numRemovedFiles -> 0, numRemovedBytes -> 0, numOutputRows -> 90848212, numOutputBytes -> 3047411070)",null,Databricks-Runtime/17.1.x-photon-scala2.13
0,2025-09-26T13:49:50.000Z,147711536460494,sumit.trendytech03@outlook.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.autoOptimize.autoCompact"":""false"",""delta.autoOptimize.optimizeWrite"":""false"",""collation"":""UTF8_BINARY"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(1701835403252703),0926-134922-hibp07z1-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/17.1.x-photon-scala2.13


In [0]:
%sql
select count(*) from demo_taxi_200files where trip_distance > 100;

count(*)
209


In [0]:
%sql
select min(trip_distance), max(trip_distance), _metadata.file_name
from demo_taxi_200files
group by _metadata.file_name
order by min(trip_distance)

min(trip_distance),max(trip_distance),file_name
0.0,57.7,part-00114-cdab6d97-d15f-47a4-8060-18deccd28000.c000.snappy.parquet
0.0,500.0,part-00070-352e1c3f-1386-491e-8bbc-455a67c6dbe5.c000.snappy.parquet
0.0,94.9,part-00136-08130224-faa8-4580-850c-0c4a3f9c01b5.c000.snappy.parquet
0.0,96.7,part-00029-f3e65714-1643-4119-af3a-ea7b229b1985.c000.snappy.parquet
0.0,813.1,part-00072-e8a5a135-b30d-4e78-908b-4b497c0639ce.c000.snappy.parquet
0.0,252.2,part-00024-a0c215f1-f18b-4663-a87a-0a8590e31f1b.c000.snappy.parquet
0.0,190.0,part-00062-7082350f-7dce-4a04-baba-30c2f0afcb19.c000.snappy.parquet
0.0,105.6,part-00043-c56d5592-a632-41c8-b8d0-071f18cff225.c000.snappy.parquet
0.0,1000000.0,part-00162-cdd32e64-c88e-4c30-bc7f-3f89680f317c.c000.snappy.parquet
0.0,570.0,part-00103-b362d33a-5881-4b47-abda-16244e9b0776.c000.snappy.parquet


In [0]:
%sql
optimize demo_taxi_200files;

path,metrics
abfss://unitycatalog@ttmystorageaccount001.dfs.core.windows.net/catalog/__unitystorage/catalogs/758280e5-e8ae-48b0-840c-9f10d52cc821/tables/48ebd9fd-3ee7-46c3-9633-d06dd87c1f5b,"List(61, 200, List(17144528, 78374073, 4.898625967213115E7, 61, 2988161840), List(15204279, 15270529, 1.523705535E7, 200, 3047411070), 0, null, null, 0, 1, 200, 0, false, 0, 0, 1758894924761, 1758894961239, 32, 1, null, List(0, 0), null, 18, 18, 243067, 0, List(3047411070, true, false, false, 0.9808961533936782, List(0.9808961533936782), 1.0, null, 0, 200, 3047411070, 3047411070, 0, 0, 0, null, log, 16777216, 67108864, 4, 0, 0, List(0), 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 3047411070, 3047411070, List(228, 1146, 0, 529, 401, 2353), 2, 1, 5, sizeAware))"
abfss://unitycatalog@ttmystorageaccount001.dfs.core.windows.net/catalog/__unitystorage/catalogs/758280e5-e8ae-48b0-840c-9f10d52cc821/tables/48ebd9fd-3ee7-46c3-9633-d06dd87c1f5b,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 61, 0, false, 0, 0, 1758894961305, 1758894969597, 24, 0, null, List(0, 0), null, 18, 18, 0, 0, List(2988161840, false, false, false, 0.9808961533936782, List(0.9808961533936782), 1.0, null, 0, 0, 0, 0, 0, 0, 0, null, log, 16777216, 67108864, 4, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(86, 47, 4825, 0, 0, 0), 2, 2, 5, sizeAware))"
abfss://unitycatalog@ttmystorageaccount001.dfs.core.windows.net/catalog/__unitystorage/catalogs/758280e5-e8ae-48b0-840c-9f10d52cc821/tables/48ebd9fd-3ee7-46c3-9633-d06dd87c1f5b,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 61, 0, true, 0, 0, 1758894969625, 1758894971749, 24, 0, null, List(0, 0), null, 18, 18, 0, 0, List(2988161840, false, false, false, 0.9808961533936782, List(0.9808961533936782), 1.0, post-optimize-compaction, 0, 0, 0, 0, 0, 0, 0, null, null, 33554432, 67108864, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(0, 0, 376, 0, 0, 0), 15, 1, 1, null))"


In [0]:
%sql
select count(*) from demo_taxi_200files where trip_distance > 100;

count(*)
209


In [0]:
%sql
select min(trip_distance), max(trip_distance), _metadata.file_name
from demo_taxi_200files
group by _metadata.file_name
order by min(trip_distance)

min(trip_distance),max(trip_distance),file_name
0.0,0.399,part-00042-fda7dc7c-ffe7-4ff4-9a82-f88310086eeb.c000.snappy.parquet
0.0,0.399,part-00058-405810bf-fd93-497a-b956-f8c73678e39c.c000.snappy.parquet
0.4,0.499,part-00051-bac14e3f-0ced-4b03-b7cf-20ae3687a6a9.c000.snappy.parquet
0.5,0.549,part-00041-6ad9f210-2734-4850-b1ce-770aa1c0cb83.c000.snappy.parquet
0.55,0.649,part-00023-30a5d0bb-9d82-42c3-b0b7-153456c57988.c000.snappy.parquet
0.65,0.709,part-00059-0b4509d0-9ffb-4d25-a9a2-3d308b00a909.c000.snappy.parquet
0.65,0.709,part-00038-94175056-edbc-4bb7-b1c6-78a9de7bb068.c000.snappy.parquet
0.71,0.749,part-00028-0405800d-14d5-4b72-8e04-0fbec38e4ebc.c000.snappy.parquet
0.75,0.799,part-00039-961aabf0-1e0d-4d77-bbe5-947aa2d8dd71.c000.snappy.parquet
0.8,0.8299999999999998,part-00060-9870d4f8-3040-4177-94db-220c48300094.c000.snappy.parquet


In [0]:
%sql
describe detail demo_taxi_200files;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,d1e90427-d20e-4068-b0fc-0311270ea75f,deltalake_catalog.default.demo_taxi_200files,null,abfss://unitycatalog@ttmystorageaccount001.dfs.core.windows.net/catalog/__unitystorage/catalogs/758280e5-e8ae-48b0-840c-9f10d52cc821/tables/48ebd9fd-3ee7-46c3-9633-d06dd87c1f5b,2025-09-26T13:49:49.591Z,2025-09-26T13:56:00.000Z,List(),List(trip_distance),61,2988161840,"Map(delta.autoOptimize.autoCompact -> false, delta.enableDeletionVectors -> true, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, delta.autoOptimize.optimizeWrite -> false, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-aa4969d1-fbae-46b5-80f7-23a1fbb5b646, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-0f9bc765-643d-47d9-b4c2-32e7d6f0abb0, collation -> UTF8_BINARY)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
ALTER TABLE demo_taxi_200files CLUSTER BY AUTO;

In [0]:
%sql
ALTER TABLE demo_taxi_200files CLUSTER BY NONE;